In [1]:
#clear all
%reset -f

#import packages
import numpy as np
import scipy
import sys
import os
import pandas as pd
import mne
import matplotlib
import h5py
from sklearn.utils import resample
from mne_icalabel import label_components

root = 'F:/Documents/Science/MirRevAdaptEEG'
participants = list(range(0,32))
#specify which erp we are analyzing
erps = 'lrp'
roi = 'latcen'

#pop up plots as separate window & interactive
%matplotlib qt
matplotlib.pyplot.close('all')

In [2]:
#setting up path/ directory
#access trial type epoched data for each participant
def load_tfr_epochs(pp_num, root_dir, erp_path, task):

    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    filename = os.path.join(pp_directory, 'p%03d_%s-epo.fif' % (pp_num, task))

    epochs = mne.read_epochs(filename)

    return epochs, pp_directory

In [3]:
#set up parameters for time-frequency morlet convolution
def set_tfr_params(l_freq = 6, h_freq = 35, num_freq = 50, cycle = 6, samp_rate = 200):
    freqs = np.logspace(*np.log10([l_freq, h_freq]), num = num_freq)
    n_cycles = cycle
    sfreq = samp_rate
    
    return freqs, n_cycles, sfreq

In [4]:
#save tfr'd data
def save_tfr_data(pp_num, root_dir, erp_path, data, task, output):
    # Save the tfr'd data
    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    out_fname = os.path.join(pp_directory, 'p%03d_%s_%s-tfr.h5' % (pp_num, task, output))
    mne.time_frequency.write_tfrs(out_fname, tfr=data, overwrite = True)

In [5]:
#load tfr'd data
def load_tfr_data(pp_num, root_dir, erp_path, task, output):
    # Save the tfr'd data
    root_directory = root_dir
    data_directory = os.path.join(root_directory, 'data/eeg/')
    id_directory = os.path.join(data_directory, 'p%03d/' % pp_num)
    pp_directory = os.path.join(id_directory, erp_path)
    out_fname = os.path.join(pp_directory, 'p%03d_%s_%s-tfr.h5' % (pp_num, task, output))
    dat = mne.time_frequency.read_tfrs(out_fname)
    
    return dat

In [6]:
# Comparing males and females would be an independent samples comparison
# permutation function should be modified
# F dist used here, but oneway F test = independent t test when dealing with two independent conditions only
def get_ind_clust_perm_test(conditionA, conditionB, pval, n_permutations, n_conditions = 2):
    #define cluster forming threshold based on p-value
    dfn = n_conditions - 1  # degrees of freedom numerator
    dfd = len(participants) - n_conditions  # degrees of freedom denominator
    thresh = scipy.stats.f.ppf(1 - pval, dfn=dfn, dfd=dfd)  
    #run cluster-based permutation test
    F_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_test([conditionA, conditionB], threshold = thresh, 
                                                          n_permutations = n_permutations, tail = 0, 
                                                          adjacency = None, seed = 999, 
                                                          out_type = 'mask', verbose = True)

    return F_0, clust_idx, clust_pvals, H0

In [ ]:
# load in tfr object, ensure to baseline correct; (tfr shape is channels, freqs, timepts)
# narrow down to channels 
# narrow down to frequencies and timepts
# take mean of theta, alpha, or beta ranges (freqs)
# take mean across electrodes (channels)

perturb_conds = ['early_late_aligned', 'early_rot', 'late_rot', 'early_mir', 'late_mir', 'early_rdm', 'late_rdm']

# lateral central CHANNELS
channels = ['C5', 'C3',
            'CP5', 'CP3', 'CP1',
            'P5', 'P3', 'P1']


# BASELINE FOR GO ONSET
baseline_t = (-1.3, -1.0)

# THETA
freq_lower = 6
freq_upper = 8

theta_latcen_EarlyLateAligned = []
theta_latcen_EarlyRot = []
theta_latcen_LateRot = []
theta_latcen_EarlyMir = []
theta_latcen_LateMir = []
theta_latcen_EarlyRdm = []
theta_latcen_LateRdm = []
full_theta_latcen_EarlyLateAligned = []
full_theta_latcen_EarlyRot = []
full_theta_latcen_LateRot = []
full_theta_latcen_EarlyMir = []
full_theta_latcen_LateMir = []
full_theta_latcen_EarlyRdm = []
full_theta_latcen_LateRdm = []

for pcond in range(0, len(perturb_conds)):
    latcenppdat = []
    full_latcenppdat = []
    for pp in participants:
        dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = perturb_conds[pcond], output = "power")
        dat = dat[0]*1e12 #convert V^2 to uV^2
        # apply baseline correction
        dat = dat.apply_baseline(baseline_t, mode='mean')
        # narrow down to channels
        dat = dat.pick_channels(channels)
        #transform to ndarray (channels, freqs, times)
        npdat = dat.data

        # get needed frequency indices
        nfreqs = [] #get indices of frequencies we want
        for i in range(0, len(dat.freqs)):
            if dat.freqs[i] >= freq_lower and dat.freqs[i] <= freq_upper:
                nfreqs.append(i)
        npdat = npdat[:, nfreqs, :]
        full_npdat = npdat #all timepts included

        # get needed timept indices
        ntimes = list(range(200,401))
        npdat = npdat[:,:,ntimes]
        
        ppdat = np.mean(npdat, axis = 1) #calculate mean across frequencies
        ppdat = np.mean(ppdat, axis = 0) # calculate mean across channels
        
        full_ppdat = np.mean(full_npdat, axis = 1)
        full_ppdat = np.mean(full_ppdat, axis = 0)
        
        latcenppdat.append(ppdat)
        full_latcenppdat.append(full_ppdat)
        
    if pcond == 0:
        theta_latcen_EarlyLateAligned.append(latcenppdat)
        full_theta_latcen_EarlyLateAligned.append(full_latcenppdat)
    elif pcond == 1:
        theta_latcen_EarlyRot.append(latcenppdat)
        full_theta_latcen_EarlyRot.append(full_latcenppdat)
    elif pcond == 2:
        theta_latcen_LateRot.append(latcenppdat)
        full_theta_latcen_LateRot.append(full_latcenppdat)
    elif pcond == 3:
        theta_latcen_EarlyMir.append(latcenppdat)
        full_theta_latcen_EarlyMir.append(full_latcenppdat)
    elif pcond == 4:
        theta_latcen_LateMir.append(latcenppdat)
        full_theta_latcen_LateMir.append(full_latcenppdat)
    elif pcond == 5:
        theta_latcen_EarlyRdm.append(latcenppdat)
        full_theta_latcen_EarlyRdm.append(full_latcenppdat)
    elif pcond == 6:
        theta_latcen_LateRdm.append(latcenppdat)
        full_theta_latcen_LateRdm.append(full_latcenppdat)

# ALPHA
freq_lower = 9
freq_upper = 13

alpha_latcen_EarlyLateAligned = []
alpha_latcen_EarlyRot = []
alpha_latcen_LateRot = []
alpha_latcen_EarlyMir = []
alpha_latcen_LateMir = []
alpha_latcen_EarlyRdm = []
alpha_latcen_LateRdm = []
full_alpha_latcen_EarlyLateAligned = []
full_alpha_latcen_EarlyRot = []
full_alpha_latcen_LateRot = []
full_alpha_latcen_EarlyMir = []
full_alpha_latcen_LateMir = []
full_alpha_latcen_EarlyRdm = []
full_alpha_latcen_LateRdm = []

for pcond in range(0, len(perturb_conds)):
    latcenppdat = []
    full_latcenppdat = []
    for pp in participants:
        dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = perturb_conds[pcond], output = "power")
        dat = dat[0]*1e12 #convert V^2 to uV^2
        # apply baseline correction
        dat = dat.apply_baseline(baseline_t, mode='mean')
        # narrow down to channels
        dat = dat.pick_channels(channels)
        #transform to ndarray (channels, freqs, times)
        npdat = dat.data

        # get needed frequency indices
        nfreqs = [] #get indices of frequencies we want
        for i in range(0, len(dat.freqs)):
            if dat.freqs[i] >= freq_lower and dat.freqs[i] <= freq_upper:
                nfreqs.append(i)
        npdat = npdat[:, nfreqs, :]
        full_npdat = npdat #all timepts included

        # get needed timept indices
        ntimes = list(range(200,401))
        npdat = npdat[:,:,ntimes]
        
        ppdat = np.mean(npdat, axis = 1) #calculate mean across frequencies
        ppdat = np.mean(ppdat, axis = 0) # calculate mean across channels
        
        full_ppdat = np.mean(full_npdat, axis = 1)
        full_ppdat = np.mean(full_ppdat, axis = 0)
        
        latcenppdat.append(ppdat)
        full_latcenppdat.append(full_ppdat)
        
    if pcond == 0:
        alpha_latcen_EarlyLateAligned.append(latcenppdat)
        full_alpha_latcen_EarlyLateAligned.append(full_latcenppdat)
    elif pcond == 1:
        alpha_latcen_EarlyRot.append(latcenppdat)
        full_alpha_latcen_EarlyRot.append(full_latcenppdat)
    elif pcond == 2:
        alpha_latcen_LateRot.append(latcenppdat)
        full_alpha_latcen_LateRot.append(full_latcenppdat)
    elif pcond == 3:
        alpha_latcen_EarlyMir.append(latcenppdat)
        full_alpha_latcen_EarlyMir.append(full_latcenppdat)
    elif pcond == 4:
        alpha_latcen_LateMir.append(latcenppdat)
        full_alpha_latcen_LateMir.append(full_latcenppdat)
    elif pcond == 5:
        alpha_latcen_EarlyRdm.append(latcenppdat)
        full_alpha_latcen_EarlyRdm.append(full_latcenppdat)
    elif pcond == 6:
        alpha_latcen_LateRdm.append(latcenppdat)
        full_alpha_latcen_LateRdm.append(full_latcenppdat)

# BETA
freq_lower = 13
freq_upper = 25

beta_latcen_EarlyLateAligned = []
beta_latcen_EarlyRot = []
beta_latcen_LateRot = []
beta_latcen_EarlyMir = []
beta_latcen_LateMir = []
beta_latcen_EarlyRdm = []
beta_latcen_LateRdm = []
full_beta_latcen_EarlyLateAligned = []
full_beta_latcen_EarlyRot = []
full_beta_latcen_LateRot = []
full_beta_latcen_EarlyMir = []
full_beta_latcen_LateMir = []
full_beta_latcen_EarlyRdm = []
full_beta_latcen_LateRdm = []

for pcond in range(0, len(perturb_conds)):
    latcenppdat = []
    full_latcenppdat = []
    for pp in participants:
        dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = perturb_conds[pcond], output = "power")
        dat = dat[0]*1e12 #convert V^2 to uV^2
        # apply baseline correction
        dat = dat.apply_baseline(baseline_t, mode='mean')
        # narrow down to channels
        dat = dat.pick_channels(channels)
        #transform to ndarray (channels, freqs, times)
        npdat = dat.data

        # get needed frequency indices
        nfreqs = [] #get indices of frequencies we want
        for i in range(0, len(dat.freqs)):
            if dat.freqs[i] >= freq_lower and dat.freqs[i] <= freq_upper:
                nfreqs.append(i)
        npdat = npdat[:, nfreqs, :]
        full_npdat = npdat #all timepts included

        # get needed timept indices
        ntimes = list(range(200,401))
        npdat = npdat[:,:,ntimes]
        
        ppdat = np.mean(npdat, axis = 1) #calculate mean across frequencies
        ppdat = np.mean(ppdat, axis = 0) # calculate mean across channels
        
        full_ppdat = np.mean(full_npdat, axis = 1)
        full_ppdat = np.mean(full_ppdat, axis = 0)
        
        latcenppdat.append(ppdat)
        full_latcenppdat.append(full_ppdat)
        
    if pcond == 0:
        beta_latcen_EarlyLateAligned.append(latcenppdat)
        full_beta_latcen_EarlyLateAligned.append(full_latcenppdat)
    elif pcond == 1:
        beta_latcen_EarlyRot.append(latcenppdat)
        full_beta_latcen_EarlyRot.append(full_latcenppdat)
    elif pcond == 2:
        beta_latcen_LateRot.append(latcenppdat)
        full_beta_latcen_LateRot.append(full_latcenppdat)
    elif pcond == 3:
        beta_latcen_EarlyMir.append(latcenppdat)
        full_beta_latcen_EarlyMir.append(full_latcenppdat)
    elif pcond == 4:
        beta_latcen_LateMir.append(latcenppdat)
        full_beta_latcen_LateMir.append(full_latcenppdat)
    elif pcond == 5:
        beta_latcen_EarlyRdm.append(latcenppdat)
        full_beta_latcen_EarlyRdm.append(full_latcenppdat)
    elif pcond == 6:
        beta_latcen_LateRdm.append(latcenppdat)
        full_beta_latcen_LateRdm.append(full_latcenppdat)
    

Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/lrp\p000_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p001/lrp\p001_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p002/lrp\p002_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p003/lrp\p003_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p004/lrp\p004_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p005/lrp\p005_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p006/lrp\p006_early_late_aligned_power-tfr.h5 ...
Applying baseline correctio

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p027/lrp\p027_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p028/lrp\p028_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p029/lrp\p029_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p030/lrp\p030_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p031/lrp\p031_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/lrp\p000_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p001/lrp\p001_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading 

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p024/lrp\p024_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p025/lrp\p025_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p026/lrp\p026_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p027/lrp\p027_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p028/lrp\p028_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p029/lrp\p029_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p030/lrp\p030_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Readin

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p021/lrp\p021_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p022/lrp\p022_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p023/lrp\p023_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p024/lrp\p024_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p025/lrp\p025_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p026/lrp\p026_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p027/lrp\p027_early_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Readin

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p017/lrp\p017_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p018/lrp\p018_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p019/lrp\p019_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p020/lrp\p020_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p021/lrp\p021_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p022/lrp\p022_early_late_aligned_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p023/lrp\p023_early_late_aligned_po

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p013/lrp\p013_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p014/lrp\p014_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p015/lrp\p015_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p016/lrp\p016_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p017/lrp\p017_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p018/lrp\p018_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p019/lrp\p019_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Do

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p010/lrp\p010_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p011/lrp\p011_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p012/lrp\p012_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p013/lrp\p013_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p014/lrp\p014_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p015/lrp\p015_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p016/lrp\p016_late_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Do

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p007/lrp\p007_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p008/lrp\p008_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p009/lrp\p009_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p010/lrp\p010_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p011/lrp\p011_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p012/lrp\p012_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p013/lrp\p013_late_rdm_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Do

Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p001/lrp\p001_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p002/lrp\p002_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p003/lrp\p003_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p004/lrp\p004_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p005/lrp\p005_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p006/lrp\p006_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p007/lrp\p007_early_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data

Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p030/lrp\p030_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p031/lrp\p031_late_rot_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p000/lrp\p000_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p001/lrp\p001_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p002/lrp\p002_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p003/lrp\p003_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading F:/Documents/Science/MirRevAdaptEEG\data/eeg/p004/lrp\p004_early_mir_power-tfr.h5 ...
Applying baseline correction (mode: mean)
Reading 

In [ ]:
# Get timepts to use for indices given by cluster-based permutation later and for timepts in saved data frames
# But only grab -0.25 to 1.5 sec time-locked to feedback onset (idx= 350:701)
root_directory = root
pp = 0 #only need one participant

# we can use aligned data
dat = load_tfr_data(pp_num = pp, root_dir = root, erp_path = erps, task = 'early_late_aligned', output = "power")
full_time = dat[0].times
time = full_time[200:401] #get only timepoints we want

In [ ]:
# Next, we subtract aligned from each condition, so that we can compare early vs late in each perturbation type

freq_bands = ['theta', 'alpha', 'beta']
diffconds = ['earlyrot', 'laterot', 'earlyrdm', 'laterdm', 'earlymir', 'latemir']

theta_latcen_earlyrot_diff = []
theta_latcen_laterot_diff = []
theta_latcen_earlyrdm_diff = []
theta_latcen_laterdm_diff = []
theta_latcen_earlymir_diff = []
theta_latcen_latemir_diff = []
full_theta_latcen_earlyrot_diff = []
full_theta_latcen_laterot_diff = []
full_theta_latcen_earlyrdm_diff = []
full_theta_latcen_laterdm_diff = []
full_theta_latcen_earlymir_diff = []
full_theta_latcen_latemir_diff = []

alpha_latcen_earlyrot_diff = []
alpha_latcen_laterot_diff = []
alpha_latcen_earlyrdm_diff = []
alpha_latcen_laterdm_diff = []
alpha_latcen_earlymir_diff = []
alpha_latcen_latemir_diff = []
full_alpha_latcen_earlyrot_diff = []
full_alpha_latcen_laterot_diff = []
full_alpha_latcen_earlyrdm_diff = []
full_alpha_latcen_laterdm_diff = []
full_alpha_latcen_earlymir_diff = []
full_alpha_latcen_latemir_diff = []

beta_latcen_earlyrot_diff = []
beta_latcen_laterot_diff = []
beta_latcen_earlyrdm_diff = []
beta_latcen_laterdm_diff = []
beta_latcen_earlymir_diff = []
beta_latcen_latemir_diff = []
full_beta_latcen_earlyrot_diff = []
full_beta_latcen_laterot_diff = []
full_beta_latcen_earlyrdm_diff = []
full_beta_latcen_laterdm_diff = []
full_beta_latcen_earlymir_diff = []
full_beta_latcen_latemir_diff = []

for band in range(0, len(freq_bands)):
    if band == 0:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(theta_latcen_EarlyRot[0], theta_latcen_EarlyLateAligned[0])
                theta_latcen_earlyrot_diff.append(diffevks)
                theta_latcen_earlyrot_diff = theta_latcen_earlyrot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_theta_latcen_EarlyRot[0], full_theta_latcen_EarlyLateAligned[0])
                full_theta_latcen_earlyrot_diff.append(full_diffevks)
                full_theta_latcen_earlyrot_diff = full_theta_latcen_earlyrot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(theta_latcen_LateRot[0], theta_latcen_EarlyLateAligned[0])
                theta_latcen_laterot_diff.append(diffevks)
                theta_latcen_laterot_diff = theta_latcen_laterot_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_LateRot[0], full_theta_latcen_EarlyLateAligned[0])
                full_theta_latcen_laterot_diff.append(full_diffevks)
                full_theta_latcen_laterot_diff = full_theta_latcen_laterot_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(theta_latcen_EarlyRdm[0], theta_latcen_EarlyLateAligned[0])
                theta_latcen_earlyrdm_diff.append(diffevks)
                theta_latcen_earlyrdm_diff = theta_latcen_earlyrdm_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_EarlyRdm[0], full_theta_latcen_EarlyLateAligned[0])
                full_theta_latcen_earlyrdm_diff.append(full_diffevks)
                full_theta_latcen_earlyrdm_diff = full_theta_latcen_earlyrdm_diff[0]
                
            elif cond == 3:
                diffevks = np.subtract(theta_latcen_LateRdm[0], theta_latcen_EarlyLateAligned[0])
                theta_latcen_laterdm_diff.append(diffevks)
                theta_latcen_laterdm_diff = theta_latcen_laterdm_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_LateRdm[0], full_theta_latcen_EarlyLateAligned[0])
                full_theta_latcen_laterdm_diff.append(full_diffevks)
                full_theta_latcen_laterdm_diff = full_theta_latcen_laterdm_diff[0]
                
            elif cond == 4:
                diffevks = np.subtract(theta_latcen_EarlyMir[0], theta_latcen_EarlyLateAligned[0])
                theta_latcen_earlymir_diff.append(diffevks)
                theta_latcen_earlymir_diff = theta_latcen_earlymir_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_EarlyMir[0], full_theta_latcen_EarlyLateAligned[0])
                full_theta_latcen_earlymir_diff.append(full_diffevks)
                full_theta_latcen_earlymir_diff = full_theta_latcen_earlymir_diff[0]
                
            elif cond == 5:
                diffevks = np.subtract(theta_latcen_LateMir[0], theta_latcen_EarlyLateAligned[0])
                theta_latcen_latemir_diff.append(diffevks)
                theta_latcen_latemir_diff = theta_latcen_latemir_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_LateMir[0], full_theta_latcen_EarlyLateAligned[0])
                full_theta_latcen_latemir_diff.append(full_diffevks)
                full_theta_latcen_latemir_diff = full_theta_latcen_latemir_diff[0]
                
    elif band == 1:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(alpha_latcen_EarlyRot[0], alpha_latcen_EarlyLateAligned[0])
                alpha_latcen_earlyrot_diff.append(diffevks)
                alpha_latcen_earlyrot_diff = alpha_latcen_earlyrot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_alpha_latcen_EarlyRot[0], full_alpha_latcen_EarlyLateAligned[0])
                full_alpha_latcen_earlyrot_diff.append(full_diffevks)
                full_alpha_latcen_earlyrot_diff = full_alpha_latcen_earlyrot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(alpha_latcen_LateRot[0], alpha_latcen_EarlyLateAligned[0])
                alpha_latcen_laterot_diff.append(diffevks)
                alpha_latcen_laterot_diff = alpha_latcen_laterot_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_LateRot[0], full_alpha_latcen_EarlyLateAligned[0])
                full_alpha_latcen_laterot_diff.append(full_diffevks)
                full_alpha_latcen_laterot_diff = full_alpha_latcen_laterot_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(alpha_latcen_EarlyRdm[0], alpha_latcen_EarlyLateAligned[0])
                alpha_latcen_earlyrdm_diff.append(diffevks)
                alpha_latcen_earlyrdm_diff = alpha_latcen_earlyrdm_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_EarlyRdm[0], full_alpha_latcen_EarlyLateAligned[0])
                full_alpha_latcen_earlyrdm_diff.append(full_diffevks)
                full_alpha_latcen_earlyrdm_diff = full_alpha_latcen_earlyrdm_diff[0]
                
            elif cond == 3:
                diffevks = np.subtract(alpha_latcen_LateRdm[0], alpha_latcen_EarlyLateAligned[0])
                alpha_latcen_laterdm_diff.append(diffevks)
                alpha_latcen_laterdm_diff = alpha_latcen_laterdm_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_LateRdm[0], full_alpha_latcen_EarlyLateAligned[0])
                full_alpha_latcen_laterdm_diff.append(full_diffevks)
                full_alpha_latcen_laterdm_diff = full_alpha_latcen_laterdm_diff[0]
                
            elif cond == 4:
                diffevks = np.subtract(alpha_latcen_EarlyMir[0], alpha_latcen_EarlyLateAligned[0])
                alpha_latcen_earlymir_diff.append(diffevks)
                alpha_latcen_earlymir_diff = alpha_latcen_earlymir_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_EarlyMir[0], full_alpha_latcen_EarlyLateAligned[0])
                full_alpha_latcen_earlymir_diff.append(full_diffevks)
                full_alpha_latcen_earlymir_diff = full_alpha_latcen_earlymir_diff[0]
                
            elif cond == 5:
                diffevks = np.subtract(alpha_latcen_LateMir[0], alpha_latcen_EarlyLateAligned[0])
                alpha_latcen_latemir_diff.append(diffevks)
                alpha_latcen_latemir_diff = alpha_latcen_latemir_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_LateMir[0], full_alpha_latcen_EarlyLateAligned[0])
                full_alpha_latcen_latemir_diff.append(full_diffevks)
                full_alpha_latcen_latemir_diff = full_alpha_latcen_latemir_diff[0]
                
    elif band == 2:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(beta_latcen_EarlyRot[0], beta_latcen_EarlyLateAligned[0])
                beta_latcen_earlyrot_diff.append(diffevks)
                beta_latcen_earlyrot_diff = beta_latcen_earlyrot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_beta_latcen_EarlyRot[0], full_beta_latcen_EarlyLateAligned[0])
                full_beta_latcen_earlyrot_diff.append(full_diffevks)
                full_beta_latcen_earlyrot_diff = full_beta_latcen_earlyrot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(beta_latcen_LateRot[0], beta_latcen_EarlyLateAligned[0])
                beta_latcen_laterot_diff.append(diffevks)
                beta_latcen_laterot_diff = beta_latcen_laterot_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_LateRot[0], full_beta_latcen_EarlyLateAligned[0])
                full_beta_latcen_laterot_diff.append(full_diffevks)
                full_beta_latcen_laterot_diff = full_beta_latcen_laterot_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(beta_latcen_EarlyRdm[0], beta_latcen_EarlyLateAligned[0])
                beta_latcen_earlyrdm_diff.append(diffevks)
                beta_latcen_earlyrdm_diff = beta_latcen_earlyrdm_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_EarlyRdm[0], full_beta_latcen_EarlyLateAligned[0])
                full_beta_latcen_earlyrdm_diff.append(full_diffevks)
                full_beta_latcen_earlyrdm_diff = full_beta_latcen_earlyrdm_diff[0]
                
            elif cond == 3:
                diffevks = np.subtract(beta_latcen_LateRdm[0], beta_latcen_EarlyLateAligned[0])
                beta_latcen_laterdm_diff.append(diffevks)
                beta_latcen_laterdm_diff = beta_latcen_laterdm_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_LateRdm[0], full_beta_latcen_EarlyLateAligned[0])
                full_beta_latcen_laterdm_diff.append(full_diffevks)
                full_beta_latcen_laterdm_diff = full_beta_latcen_laterdm_diff[0]
                
            elif cond == 4:
                diffevks = np.subtract(beta_latcen_EarlyMir[0], beta_latcen_EarlyLateAligned[0])
                beta_latcen_earlymir_diff.append(diffevks)
                beta_latcen_earlymir_diff = beta_latcen_earlymir_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_EarlyMir[0], full_beta_latcen_EarlyLateAligned[0])
                full_beta_latcen_earlymir_diff.append(full_diffevks)
                full_beta_latcen_earlymir_diff = full_beta_latcen_earlymir_diff[0]
                
            elif cond == 5:
                diffevks = np.subtract(beta_latcen_LateMir[0], beta_latcen_EarlyLateAligned[0])
                beta_latcen_latemir_diff.append(diffevks)
                beta_latcen_latemir_diff = beta_latcen_latemir_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_LateMir[0], full_beta_latcen_EarlyLateAligned[0])
                full_beta_latcen_latemir_diff.append(full_diffevks)
                full_beta_latcen_latemir_diff = full_beta_latcen_latemir_diff[0]

In [ ]:
# Next step is to subtract early from late condition, to generate a single signal for each perturbation
diffconds = ['rot', 'rdm', 'mir']
freq_bands = ['theta', 'alpha', 'beta']

theta_latcen_rot_diff = []
theta_latcen_rdm_diff = []
theta_latcen_mir_diff = []
full_theta_latcen_rot_diff = []
full_theta_latcen_rdm_diff = []
full_theta_latcen_mir_diff = []

alpha_latcen_rot_diff = []
alpha_latcen_rdm_diff = []
alpha_latcen_mir_diff = []
full_alpha_latcen_rot_diff = []
full_alpha_latcen_rdm_diff = []
full_alpha_latcen_mir_diff = []

beta_latcen_rot_diff = []
beta_latcen_rdm_diff = []
beta_latcen_mir_diff = []
full_beta_latcen_rot_diff = []
full_beta_latcen_rdm_diff = []
full_beta_latcen_mir_diff = []

for band in range(0, len(freq_bands)):
    if band == 0:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(theta_latcen_laterot_diff, theta_latcen_earlyrot_diff)
                theta_latcen_rot_diff.append(diffevks)
                theta_latcen_rot_diff = theta_latcen_rot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_theta_latcen_laterot_diff, full_theta_latcen_earlyrot_diff)
                full_theta_latcen_rot_diff.append(full_diffevks)
                full_theta_latcen_rot_diff = full_theta_latcen_rot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(theta_latcen_laterdm_diff, theta_latcen_earlyrdm_diff)
                theta_latcen_rdm_diff.append(diffevks)
                theta_latcen_rdm_diff = theta_latcen_rdm_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_laterdm_diff, full_theta_latcen_earlyrdm_diff)
                full_theta_latcen_rdm_diff.append(full_diffevks)
                full_theta_latcen_rdm_diff = full_theta_latcen_rdm_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(theta_latcen_latemir_diff, theta_latcen_earlymir_diff)
                theta_latcen_mir_diff.append(diffevks)
                theta_latcen_mir_diff = theta_latcen_mir_diff[0]
                
                full_diffevks = np.subtract(full_theta_latcen_latemir_diff, full_theta_latcen_earlymir_diff)
                full_theta_latcen_mir_diff.append(full_diffevks)
                full_theta_latcen_mir_diff = full_theta_latcen_mir_diff[0]
                
    elif band == 1:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(alpha_latcen_laterot_diff, alpha_latcen_earlyrot_diff)
                alpha_latcen_rot_diff.append(diffevks)
                alpha_latcen_rot_diff = alpha_latcen_rot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_alpha_latcen_laterot_diff, full_alpha_latcen_earlyrot_diff)
                full_alpha_latcen_rot_diff.append(full_diffevks)
                full_alpha_latcen_rot_diff = full_alpha_latcen_rot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(alpha_latcen_laterdm_diff, alpha_latcen_earlyrdm_diff)
                alpha_latcen_rdm_diff.append(diffevks)
                alpha_latcen_rdm_diff = alpha_latcen_rdm_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_laterdm_diff, full_alpha_latcen_earlyrdm_diff)
                full_alpha_latcen_rdm_diff.append(full_diffevks)
                full_alpha_latcen_rdm_diff = full_alpha_latcen_rdm_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(alpha_latcen_latemir_diff, alpha_latcen_earlymir_diff)
                alpha_latcen_mir_diff.append(diffevks)
                alpha_latcen_mir_diff = alpha_latcen_mir_diff[0]
                
                full_diffevks = np.subtract(full_alpha_latcen_latemir_diff, full_alpha_latcen_earlymir_diff)
                full_alpha_latcen_mir_diff.append(full_diffevks)
                full_alpha_latcen_mir_diff = full_alpha_latcen_mir_diff[0]
                
    elif band == 2:
        for cond in range(0, len(diffconds)):
            if cond == 0:
                diffevks = np.subtract(beta_latcen_laterot_diff, beta_latcen_earlyrot_diff)
                beta_latcen_rot_diff.append(diffevks)
                beta_latcen_rot_diff = beta_latcen_rot_diff[0] #to keep shape of object consistent
                
                full_diffevks = np.subtract(full_beta_latcen_laterot_diff, full_beta_latcen_earlyrot_diff)
                full_beta_latcen_rot_diff.append(full_diffevks)
                full_beta_latcen_rot_diff = full_beta_latcen_rot_diff[0] #to keep shape of object consistent
                
            elif cond == 1:
                diffevks = np.subtract(beta_latcen_laterdm_diff, beta_latcen_earlyrdm_diff)
                beta_latcen_rdm_diff.append(diffevks)
                beta_latcen_rdm_diff = beta_latcen_rdm_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_laterdm_diff, full_beta_latcen_earlyrdm_diff)
                full_beta_latcen_rdm_diff.append(full_diffevks)
                full_beta_latcen_rdm_diff = full_beta_latcen_rdm_diff[0]
                
            elif cond == 2:
                diffevks = np.subtract(beta_latcen_latemir_diff, beta_latcen_earlymir_diff)
                beta_latcen_mir_diff.append(diffevks)
                beta_latcen_mir_diff = beta_latcen_mir_diff[0]
                
                full_diffevks = np.subtract(full_beta_latcen_latemir_diff, full_beta_latcen_earlymir_diff)
                full_beta_latcen_mir_diff.append(full_diffevks)
                full_beta_latcen_mir_diff = full_beta_latcen_mir_diff[0]

In [ ]:
#separate by sex

#beta
beta_latcen_rot_males = beta_latcen_rot_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
beta_latcen_rot_females = beta_latcen_rot_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

beta_latcen_rdm_males = beta_latcen_rdm_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
beta_latcen_rdm_females = beta_latcen_rdm_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

beta_latcen_mir_males = beta_latcen_mir_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
beta_latcen_mir_females = beta_latcen_mir_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

#alpha
alpha_latcen_rot_males = alpha_latcen_rot_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
alpha_latcen_rot_females = alpha_latcen_rot_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

alpha_latcen_rdm_males = alpha_latcen_rdm_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
alpha_latcen_rdm_females = alpha_latcen_rdm_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

alpha_latcen_mir_males = alpha_latcen_mir_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
alpha_latcen_mir_females = alpha_latcen_mir_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

#theta
theta_latcen_rot_males = theta_latcen_rot_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
theta_latcen_rot_females = theta_latcen_rot_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

theta_latcen_rdm_males = theta_latcen_rdm_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
theta_latcen_rdm_females = theta_latcen_rdm_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

theta_latcen_mir_males = theta_latcen_mir_diff[[0,3,4,5,8,12,14,15,16,17,19,20,23,24,26,27,28,30,31]]
theta_latcen_mir_females = theta_latcen_mir_diff[[1,2,6,7,9,10,11,13,18,21,22,25,29]]

In [ ]:
# Compare EARLY VS LATE for each perturbation
# Generate a data frame to tabulate condition, cluster indices, cluster timepts, p values
# This information can then be included in plots
p = 0.05
perms = 1000

condition = []
clust_idx_start = []
clust_idx_end = []
time_start = []
time_end = []
p_values = []

freq_bands = ['theta', 'alpha', 'beta']

for band in range(0, len(freq_bands)):
    if band == 0:
        theta_conditionnames = ['theta_latcen_rot_mvf', 'theta_latcen_rdm_mvf', 'theta_latcen_mir_mvf']
        for c in range(0, len(theta_conditionnames)):
            if c == 0:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(theta_latcen_rot_males, theta_latcen_rot_females, p, perms)
        #         print(clust_idx, clust_pvals)
                if len(clust_idx) == 0:
                    condition.append(theta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(theta_conditionnames[c])
            
            elif c == 1:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(theta_latcen_rdm_males, theta_latcen_rdm_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(theta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(theta_conditionnames[c])
            
            elif c == 2:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(theta_latcen_mir_males, theta_latcen_mir_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(theta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(theta_conditionnames[c])
    elif band == 1:
        alpha_conditionnames = ['alpha_latcen_rot_mvf', 'alpha_latcen_rdm_mvf', 'alpha_latcen_mir_mvf']
        for c in range(0, len(alpha_conditionnames)):
            if c == 0:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(alpha_latcen_rot_males, alpha_latcen_rot_females, p, perms)
        #         print(clust_idx, clust_pvals)
                if len(clust_idx) == 0:
                    condition.append(alpha_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(alpha_conditionnames[c])
            
            elif c == 1:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(alpha_latcen_rdm_males, alpha_latcen_rdm_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(alpha_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(alpha_conditionnames[c])
            
            elif c == 2:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(alpha_latcen_mir_males, alpha_latcen_mir_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(alpha_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(alpha_conditionnames[c])
            
    elif band == 2:
        beta_conditionnames = ['beta_latcen_rot_mvf', 'beta_latcen_rdm_mvf', 'beta_latcen_mir_mvf']
        for c in range(0, len(beta_conditionnames)):
            if c == 0:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(beta_latcen_rot_males, beta_latcen_rot_females, p, perms)
        #         print(clust_idx, clust_pvals)
                if len(clust_idx) == 0:
                    condition.append(beta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(beta_conditionnames[c])
            
            elif c == 1:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(beta_latcen_rdm_males, beta_latcen_rdm_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(beta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(beta_conditionnames[c])
            
            elif c == 2:
                T_0, clust_idx, clust_pvals, H0 = get_ind_clust_perm_test(beta_latcen_mir_males, beta_latcen_mir_females, p, perms)
                if len(clust_idx) == 0:
                    condition.append(beta_conditionnames[c])
                    clust_idx_start.append(np.nan)
                    clust_idx_end.append(np.nan)
                    time_start.append(np.nan)
                    time_end.append(np.nan)
                    p_values.append(np.nan)
                else:    
                    for clust in range(0, len(clust_idx)):
                        cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                        cluster_start = cluster.start
                        clust_idx_start.append(cluster_start)
        
                        cluster_end = cluster.stop
                        clust_idx_end.append(cluster_end)
        
                        time_idx_start = time[cluster_start]
                        time_start.append(time_idx_start)
        
                        time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                        time_end.append(time_idx_end)
        
                        clust_p = clust_pvals[clust]
                        p_values.append(clust_p)
        
                        condition.append(beta_conditionnames[c])
        
perm_test = pd.DataFrame(
    {'condition': condition,
     'clust_idx_start': clust_idx_start,
     'clust_idx_end': clust_idx_end,
     'time_start': time_start,
     'time_end': time_end,
     'p_values': p_values})

perm_test_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/sex_diff/', 'TFR_Permutation_test_EvL_PerturbTypeComp_SexDIFF_%s_%s.csv' % (erps, roi))
perm_test.to_csv(perm_test_filename)